#  Curso: Flushes y Gestión de Prompts en Langfuse

Dos temas que preparan el terreno para LLMOps:

- **PARTE A — Flushes:** observar *en números* la diferencia entre el envío por lotes
  (async, por defecto) y el envío inmediato, y entender cuándo usar cada uno.
- **PARTE B — Prompts:** sacar los prompts del código y gestionarlos versionados en Langfuse
  (crear, versionar, etiquetar, recuperar, compilar y enlazar a la generación). Es la base de
  la Clase 9 (versionado y A/B testing de prompts).

### Prerrequisito
Mismo setup de Langfuse de las clases anteriores (mismas claves).


## 1. Instalación y conexión

In [ ]:
!pip install transformers torch huggingface_hub rank_bm25 -q
!pip install langgraph langfuse -q

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os, time
os.environ["LANGFUSE_HOST"]       = "https://us.cloud.langfuse.com"
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-8c53a053-7a99-4afe-9d78-ffc5ecc10432"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-ef93aaf8-901b-4c6b-83e2-ce51c55d9f1c"
# 🔒 rota la secret key cuando termines de probar

from langfuse import get_client, propagate_attributes
langfuse = get_client()
print("✅ Conectado" if langfuse.auth_check() else "❌ auth_check fallo")

✅ Conectado


---
# PARTE A — Flushes: por lotes vs inmediato

Recordatorio: el SDK **encola** los eventos y los envía en segundo plano, en lotes que se
disparan por **tiempo** (`flush_interval`) o **tamaño** (`flush_at`, por defecto 512 spans).
`flush()` fuerza el envío del lote pendiente ahora mismo.

Vamos a *medir* el efecto de forzar el envío en cada traza (cuasi tiempo real) frente a dejar
que el SDK agrupe. Usamos spans vacíos (sin LLM) para aislar el coste del envío.

### A1 — Por lotes (comportamiento por defecto): crear no bloquea

In [ ]:
N = 15

# Crear los spans: se encolan, NO se envian aqui -> deberia ser casi instantaneo
t0 = time.perf_counter()
for i in range(N):
    with langfuse.start_as_current_observation(as_type="span", name=f"batched-{i}") as s:
        s.update(input={"i": i})
t_crear = (time.perf_counter() - t0) * 1000

# Un unico flush envia TODO el lote de una vez
t0 = time.perf_counter()
langfuse.flush()
t_flush_unico = (time.perf_counter() - t0) * 1000

print(f"Crear {N} spans (en tu 'camino critico'): {t_crear:6.1f} ms  ->  {t_crear/N:.2f} ms/traza")
print(f"Flush unico de todo el lote:            {t_flush_unico:6.1f} ms")

Crear 15 spans (en tu 'camino critico'):   43.2 ms  ->  2.88 ms/traza
Flush unico de todo el lote:             248.6 ms


### A2 — Inmediato (`flush()` por traza): cuasi tiempo real, pero bloquea

In [ ]:
# Forzamos el envio en cada iteracion: cada traza espera su ida y vuelta de red
t0 = time.perf_counter()
for i in range(N):
    with langfuse.start_as_current_observation(as_type="span", name=f"realtime-{i}") as s:
        s.update(input={"i": i})
    langfuse.flush()   # 👈 envio inmediato en cada traza
t_rt = (time.perf_counter() - t0) * 1000

print(f"{N} spans con flush por traza: {t_rt:7.1f} ms  ->  {t_rt/N:.2f} ms/traza")
print(f"\nComparativa (tiempo dentro de tu path por traza):")
print(f"  A1 por lotes:  ~{t_crear/N:.2f} ms/traza   (el envio ocurre fuera del path)")
print(f"  A2 inmediato:  ~{t_rt/N:.2f} ms/traza   (cada traza paga la red)")

15 spans con flush por traza:  1309.7 ms  ->  87.31 ms/traza

Comparativa (tiempo dentro de tu path por traza):
  A1 por lotes:  ~2.88 ms/traza   (el envio ocurre fuera del path)
  A2 inmediato:  ~87.31 ms/traza   (cada traza paga la red)


### A3 — Configurar el batching (`flush_at` / `flush_interval`)

No hace falta llamar `flush()` a mano para acercarte a tiempo real: puedes ajustar los
disparadores del lote. Se configuran **al inicializar el cliente**, o por variables de entorno
antes de crear el cliente:

```python
# Opcion 1: variables de entorno (ANTES del primer get_client)
os.environ["LANGFUSE_FLUSH_AT"] = "1"        # enviar cada evento de inmediato
os.environ["LANGFUSE_FLUSH_INTERVAL"] = "1"  # ...o como muy tarde cada 1 s

# Opcion 2: al construir el cliente
# from langfuse import Langfuse
# langfuse = Langfuse(flush_at=1, flush_interval=1)
```

> ⚠️ **`flush_at=1` NO es recomendable en producción.** Le suma una ida y vuelta de red a cada
> request, y con tráfico alto chocas con el límite de tasa (1000 batches/min en Hobby/Pro).
> El batching async existe precisamente para mantener la observabilidad **fuera del camino
> crítico**. Para monitoreo, el intervalo por defecto ya te da las trazas en la UI en 1–2 s.

**Cuándo forzar `flush()`:** solo en los *bordes* — al cerrar la app, en un worker que
termina, en serverless (Lambda) o, como en estos notebooks, al final de una celda de Colab
(el proceso es efímero y perderías el último lote).

---
# PARTE B — Gestión de Prompts

Sacamos el prompt de `cloudbox_bot` fuera del código y lo gestionamos en Langfuse:
versionado, etiquetas, y enlace a la generación.

### B0 — Cargar el LLM y el retriever (para generar de verdad)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("🔄 Loading LLM...")
model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto")
llm = pipeline("text-generation", model=model, tokenizer=tokenizer)
def ask_llm(prompt, max_new_tokens=60):
    out = llm([{"role": "user", "content": prompt}], max_new_tokens=max_new_tokens,
              do_sample=False, pad_token_id=tokenizer.eos_token_id, return_full_text=False)
    return out[0]["generated_text"].strip()
def count_tokens(text): return len(tokenizer.encode(text))

CORPUS = [
    {"id": "doc-02", "text": "El plan Pro de CloudBox cuesta 9 dolares al mes e incluye 2 TB."},
    {"id": "doc-03", "text": "Para restaurar un archivo borrado, ve a la Papelera; se guardan 30 dias."},
    {"id": "doc-04", "text": "CloudBox cifra los archivos en reposo con AES-256 y en transito con TLS."},
    {"id": "doc-06", "text": "El limite por archivo es 50 GB en Pro y 5 GB en Free."},
]
_bm25 = BM25Okapi([d["text"].lower().split() for d in CORPUS])
def retrieve(query, k=2):
    s = _bm25.get_scores(query.lower().split())
    return [CORPUS[i] for i in np.argsort(s)[::-1][:k]]
print("✅ Listo")

🔄 Loading LLM...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

✅ Listo


### B1 — Crear el prompt (versión 1, etiqueta `production`)

El template usa la sintaxis de Langfuse con dobles llaves: `{{context}}` y `{{question}}`.
El `config` guarda metadatos del prompt (modelo, temperatura) junto a él.

In [ ]:
prompt_v1 = """Responde la pregunta del usuario usando UNICAMENTE el contexto.
Si la respuesta no esta en el contexto, di exactamente: "No tengo esa informacion".

Contexto:
{{context}}

Pregunta: {{question}}
Respuesta:"""

langfuse.create_prompt(
    name="cloudbox-answer",
    type="text",
    prompt=prompt_v1,
    labels=["production"],                      # esta version es la de produccion
    config={"model": "gemma-3-1b-it", "temperature": 0.0},
)
print("✅ Prompt 'cloudbox-answer' v1 creado con label 'production'")
print("   (si vuelves a correr esta celda, Langfuse crea una version nueva, no falla)")

✅ Prompt 'cloudbox-answer' v1 creado con label 'production'
   (si vuelves a correr esta celda, Langfuse crea una version nueva, no falla)


### B2 — Crear una versión 2 (etiqueta `staging`)

Simulamos una mejora: un prompt más estricto. Le ponemos la etiqueta `staging`, así
**`production` sigue apuntando a la v1** y la v2 queda lista para probar sin afectar a nadie.

In [ ]:
prompt_v2 = """Eres el asistente de soporte de CloudBox. Responde de forma breve y precisa,
usando SOLO el contexto. Si no esta en el contexto, di: "No tengo esa informacion".
No inventes precios ni caracteristicas.

Contexto:
{{context}}

Pregunta del cliente: {{question}}
Respuesta:"""

langfuse.create_prompt(
    name="cloudbox-answer",
    type="text",
    prompt=prompt_v2,
    labels=["staging"],                          # NO toca production
    config={"model": "gemma-3-1b-it", "temperature": 0.0},
)
print("✅ v2 creada con label 'staging' (production sigue en v1)")

✅ v2 creada con label 'staging' (production sigue en v1)


### B3 — Recuperar y compilar el prompt en runtime

In [ ]:
# Sin argumentos -> trae la version con label 'production' (v1)
p_prod = langfuse.get_prompt("cloudbox-answer")
print(f"production -> version {p_prod.version}")

# Por label -> la de staging (v2)
p_stag = langfuse.get_prompt("cloudbox-answer", label="staging")
print(f"staging    -> version {p_stag.version}")

# Por numero de version explicito
p_v1 = langfuse.get_prompt("cloudbox-answer", version=1)
print(f"version=1  -> version {p_v1.version}\n")

# Compilar rellena las variables {{...}}
ejemplo = p_prod.compile(context="- El plan Pro cuesta 9 dolares al mes.",
                         question="¿Cuanto cuesta el plan Pro?")
print("Prompt compilado (production):\n", ejemplo)

production -> version 9
staging    -> version 10
version=1  -> version 1

Prompt compilado (production):
 Responde la pregunta del usuario usando UNICAMENTE el contexto.
Si la respuesta no esta en el contexto, di exactamente: "No tengo esa informacion".

Contexto:
- El plan Pro cuesta 9 dolares al mes.

Pregunta: ¿Cuanto cuesta el plan Pro?
Respuesta:


### B4 — Usar el prompt en una generación y **enlazarlo**

Al pasar el objeto `prompt` a la observación `generation`, Langfuse **liga esa generación con
la versión de prompt** que la produjo. En la UI verás qué versión generó cada respuesta — la
base para comparar versiones (Clase 9).

In [ ]:
PRICE_IN, PRICE_OUT = 0.0005, 0.0015

def cloudbox_bot_prompted(query, label="production"):
    cb_prompt = langfuse.get_prompt("cloudbox-answer", label=label)   # cache client-side
    with langfuse.start_as_current_observation(as_type="span", name="cloudbox_bot") as root:
        root.update(input={"query": query, "prompt_label": label})
        with propagate_attributes(trace_name="cloudbox_bot", session_id="extra-prompts"):
            docs = retrieve(query)
            contexto = "\n".join(f"- {d['text']}" for d in docs)
            compiled = cb_prompt.compile(context=contexto, question=query)

            with langfuse.start_as_current_observation(
                    as_type="generation", name="generate", model="gemma-3-1b-it",
                    prompt=cb_prompt) as gen:      # 👈 enlaza la version de prompt
                gen.update(input=compiled)
                answer = ask_llm(compiled)
                n_in, n_out = count_tokens(compiled), count_tokens(answer)
                gen.update(output=answer,
                           usage_details={"input": n_in, "output": n_out},
                           cost_details={"total": n_in/1000*PRICE_IN + n_out/1000*PRICE_OUT})
        root.update(output={"answer": answer})
    return answer

# Genera con production (v1) y con staging (v2): compararas ambas en la UI
for lbl in ["production", "staging"]:
    ans = cloudbox_bot_prompted("¿Cuanto cuesta el plan Pro?", label=lbl)
    print(f"[{lbl}] {ans}\n")

langfuse.flush()
print("✅ En Langfuse: cada generacion muestra la version de prompt enlazada.")

[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[production] 9 dolares al mes

[staging] 9 dólares al mes.

✅ En Langfuse: cada generacion muestra la version de prompt enlazada.


### B5 — Caching y fallback (por qué no añade latencia)

`get_prompt` cachea del lado del cliente, así que tras la primera llamada es lectura de
memoria. Puedes controlar el TTL y dar un `fallback` local para alta disponibilidad:

```python
# TTL de cache en segundos (por defecto ~60 s)
p = langfuse.get_prompt("cloudbox-answer", cache_ttl_seconds=300)

# Fallback local si Langfuse no responde (la app nunca se cae por el prompt)
p = langfuse.get_prompt(
    "cloudbox-answer",
    fallback="Responde usando el contexto: {{context}}\\nPregunta: {{question}}",
)
```

> ⚠️ Con `fallback`, el enlace prompt↔traza no se crea (el fallback es puramente local); es una
> red de seguridad, no el camino normal.

---
## 🎯 Cierre y puente a la Clase 9

- **Flushes:** ya sabes elegir entre lote (async, por defecto, fuera del path) e inmediato
  (`flush()` o `flush_at=1`, solo en los bordes o casos justificados).
- **Prompts:** tu prompt ya vive versionado en Langfuse, con etiquetas `production`/`staging`
  y enlazado a cada generación.

La **Clase 9** parte justo de aquí: con dos versiones (`production` vs `staging`) enlazadas a
sus generaciones, puedes hacer **A/B testing** (comparar calidad/costo por versión) y
**regression testing** (verificar que la v2 no rompe lo que la v1 ya respondía bien) — y
desplegar moviendo la etiqueta `production`, sin tocar el código.

### 🏋️ Ejercicios
1. Baja `flush_at` con variables de entorno, reinicia el kernel, y observa en la UI cómo
   cambia la rapidez con que aparecen las trazas.
2. Crea una v3 del prompt y muévela a `production`. Sin cambiar el código de
   `cloudbox_bot_prompted`, la siguiente ejecución ya la usa. Esa es la magia de los labels.
3. Corre 5 consultas con `production` y 5 con `staging`, adjunta un `create_score` de calidad
   a cada una (Clase 7), y compara la media por versión de prompt. Acabas de hacer un A/B test.
